In [ ]:
# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# %% ===================================================================
# 0. CÀRREGA I NETEJA DE DADES
# =======================================================================
DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)
top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')


# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )




# Q1

In [ ]:

df_comarques = df.groupby('Nom comarca', as_index=False).agg(
    Superficie_Total_Calcinada=('SuperficieTotal', 'sum'),
    Nombre_Incendis=('SuperficieTotal', 'count'),
    Mida_Mitjana=('SuperficieTotal', 'mean')
).round(2)

df_comarques_complet = pd.DataFrame({'Nom comarca': sorted(list(noms_geojson))}).merge(
    df_comarques, on='Nom comarca', how='left'
).fillna({'Superficie_Total_Calcinada': 0, 'Nombre_Incendis': 0, 'Mida_Mitjana': 0})

w1, h1 = mida_mapa(540)

mapa_q1_inc = alt.Chart(comarques_geo).mark_geoshape(
    stroke='white', strokeWidth=0.9
).transform_lookup(
    lookup='properties.NOMCOMAR',
    from_=alt.LookupData(
        df_comarques_complet, 'Nom comarca',
        ['Superficie_Total_Calcinada', 'Nombre_Incendis', 'Mida_Mitjana']
    )
).encode(
    color=alt.Color(
        'Nombre_Incendis:Q',
        scale=alt.Scale(scheme='oranges'),
        title="Nombre d'incendis",
        legend=alt.Legend(orient='bottom', gradientLength=260, titleFontSize=13, labelFontSize=12)
    ),
    tooltip=[
        alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
        alt.Tooltip('Nombre_Incendis:Q', title="Nombre d'incendis", format=','),
        alt.Tooltip('Superficie_Total_Calcinada:Q', title='Superfície calcinada (ha)', format=',.1f')
    ]
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
    width=w1, height=h1,
    title=alt.TitleParams("Freqüència: Nombre total d'incendis per comarca", fontSize=15, fontWeight='bold')
)

mapa_q1_ha = alt.Chart(comarques_geo).mark_geoshape(
    stroke='white', strokeWidth=0.9
).transform_lookup(
    lookup='properties.NOMCOMAR',
    from_=alt.LookupData(
        df_comarques_complet, 'Nom comarca',
        ['Superficie_Total_Calcinada', 'Nombre_Incendis', 'Mida_Mitjana']
    )
).encode(
    color=alt.Color(
        'Superficie_Total_Calcinada:Q',
        scale=alt.Scale(scheme='reds', type='symlog', constant=50),
        title='Hectàrees calcinades (ha)',
        legend=alt.Legend(orient='bottom', gradientLength=260, titleFontSize=13, labelFontSize=12)
    ),
    tooltip=[
        alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
        alt.Tooltip('Superficie_Total_Calcinada:Q', title='Superfície calcinada (ha)', format=',.1f'),
        alt.Tooltip('Nombre_Incendis:Q', title="Nombre d'incendis", format=',')
    ]
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
    width=w1, height=h1,
    title=alt.TitleParams("Impacte: Hectàrees totals calcinades per comarca", fontSize=15, fontWeight='bold')
)

q1_v1 = alt.hconcat(mapa_q1_inc, mapa_q1_ha, spacing=24).resolve_scale(color='independent').properties(
    title=titol_rt(
        "Q1 (v1) · Freqüència vs. Impacte: on s'encenen més focs no sempre és on es crema més bosc",
        "Comparativa per comarca del nombre total d'incendis (esquerra) i les hectàrees calcinades (dreta) a Catalunya (1998-2024)"
    )
).configure_view(stroke=None)
q1_v1
q1_v1.save('outputs/Q1V1.json')

# Q2

In [ ]:
"""Q2 - Evolution of the causes of fires: five Altair proposals.

Input: the cleaned fires table (OpenRefine export) with at least the columns
    Campania, Origen, Causa, SuperficieTotalForestal
Usage in Colab / Streamlit:
    from q2_charts import prepare, chart1, chart2, chart3, chart4, chart5
    d = prepare(df)            # df = your cleaned DataFrame
    chart1(d)                  # etc.
"""
import altair as alt
import pandas as pd


NAMED_CAUSA_COLORS = {
    "Intentional": "tomato",
    "Other unintentional causes": "darkorange",
    "Agricultural or livestock burning": "mediumseagreen",
    "Power lines": "gold",
    "Lightning": "royalblue",
    "Smokers": "goldenrod",
    "Unknown": "mediumpurple",
    "Other": "darkgray",
}

CAUSA_EN = {
    "Intencionado": "Intentional",
    "Altres": "Other",
    "Desconocida": "Unknown",
    "Quemas agrícolas y ganaderas": "Agricultural or livestock burning",
    "Rayo": "Lightning",
    "Otras causas no intencionales": "Other unintentional causes",
    "Causas por líneas eléctricas": "Power lines",
    "Fumadores": "Smokers",
}
CAUSA_ORDER = [
    "Intentional",
    "Other unintentional causes",
    "Agricultural or livestock burning",
    "Power lines",
    "Lightning",
    "Smokers",
    "Unknown",
    "Other",
]

LEGEND = alt.Legend(
    title="Cause of fire",
    orient="top",
    direction="horizontal",
    labelExpr="{" + ", ".join(f"'{k}': '{v}'" for k, v in CAUSA_EN.items()) + "}[datum.label] || datum.label",
)


def beeswarm_data(d, per_dot=5, cols=7, spread=0.9, layout="stacked"):
    """One row per dot (1 dot = `per_dot` fires; cumulative rounding per year)."""
    dx = spread / cols
    mapped = d["CausaAgrupada"].dropna().astype(str).map(lambda c: CAUSA_EN.get(c, c))

    freq_order = mapped.value_counts().index.tolist()
    ordered_causes = [c for c in CAUSA_ORDER if c in freq_order] + [
        c for c in freq_order if c not in CAUSA_ORDER
    ]

    temp = d.loc[mapped.index, ["Any"]].copy()
    temp["cause_en"] = mapped

    counts_all = (
        temp.groupby(["Any", "cause_en"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=ordered_causes, fill_value=0)
    )

    dots = {}
    for yr, counts in counts_all.iterrows():
        cum, prev = counts.cumsum(), 0
        for causa in ordered_causes:
            end = int(round(cum[causa] / per_dot))
            dots[(yr, causa)] = (end - prev, int(counts[causa]), prev)
            prev = end

    year_totals = {}
    for (yr, _causa), (_nd, fires, _first) in dots.items():
        year_totals[yr] = year_totals.get(yr, 0) + fires

    lane_rows = max((-(-nd // cols) for nd, _, _ in dots.values()), default=1)
    lane_h = lane_rows + 2

    out = []
    for (yr, causa), (nd, fires, first) in dots.items():
        for j in range(nd):
            k = first + j
            r, c = divmod(k, cols)
            x = yr + (c - (cols - 1) / 2) * dx + (dx / 4 if r % 2 == 0 else -dx / 4)
            y = r * 0.866
            total_yr = year_totals.get(yr, 0)
            percent_yr = (fires / total_yr) if total_yr > 0 else 0.0

            out.append({
                "Any": yr,
                "cause": causa,
                "x": x,
                "y": y,
                "fires": int(fires),
                "year_percent": percent_yr,
            })

    out_df = pd.DataFrame(out)
    out_df.attrs["lane_h"] = lane_h
    out_df.attrs["causes"] = ordered_causes
    return out_df


def chart6(d, per_dot=5, cols=7, width=1500, layout="stacked"):
    """Wide beeswarm of fires per year, coloured by cause. 1 dot = `per_dot` fires."""
    b = beeswarm_data(d, per_dot, cols, layout=layout)

    causes = b.attrs["causes"]
    cause_scale = alt.Scale(
        domain=causes,
        range=[NAMED_CAUSA_COLORS.get(c, "slategray") for c in causes],
    )
    legend = alt.Legend(
        title="Cause of fire",
        orient="top",
        direction="horizontal",
    )
    yr_min = int(d["Any"].min())
    yr_max = int(d["Any"].max())
    n_years = max(yr_max - yr_min + 1, 1)

    px_year = width / n_years
    dx_px = px_year * 0.9 / cols
    ymax = float(b["y"].max()) + 1 if not b.empty else 1.0
    height = max(int(ymax * dx_px) + 30, 120)
    size = (dx_px * 0.92) ** 2

    xenc = alt.X(
        "x:Q",
        title="Year",
        scale=alt.Scale(domain=[yr_min - 0.5, yr_max + 0.5], nice=False, zero=False),
        axis=alt.Axis(format="d", values=list(range(yr_min, yr_max + 1)), grid=False, labelFontSize=11),
    )
    title = alt.TitleParams(text=f"Fires per year by cause (1 dot = {per_dot} fires)", anchor="middle")

    yenc = alt.Y(
        "y:Q",
        title="Fires per year (approx.)",
        scale=alt.Scale(domain=[-1, ymax], nice=False),
        axis=alt.Axis(
            grid=True,
            gridDash=[2, 3],
            labelExpr=f"round(datum.value / 0.866 * {cols * per_dot})",
            values=[0.866 * v / (cols * per_dot) for v in range(0, int(ymax / 0.866 * cols * per_dot) + 1, 100 if per_dot <= 5 else 200)],
        ),
    )

    dots = (
        alt.Chart(b)
        .mark_circle(size=size, opacity=1, stroke="white", strokeWidth=0.4)
        .encode(
            x=xenc,
            y=yenc,
            color=alt.Color("cause:N", scale=cause_scale, legend=LEGEND),
            tooltip=[
                alt.Tooltip("Any:O", title="Year"),
                alt.Tooltip("cause:N", title="Cause of fire"),
                alt.Tooltip("fires:Q", title="Fires of this cause"),
                alt.Tooltip("year_percent:Q", title="% of year's total", format=".1%"),
            ],
        )
    )
    return dots.properties(width=width, height=height, title=title)


q2 = chart6(df, per_dot=5, cols=6, width=1900)
q2.save("outputs/Q2.json")
q2


# Q3

In [ ]:
# %% ===================================================================
# Q3 · 9 Mapes (1r amb escala absoluta pròpia + 8 amb diferència absoluta vs. trienni anterior)
# =======================================================================
df_q3_comarca = (
    df.groupby(['PeriodeQ3', 'Nom comarca'], as_index=False, observed=True)
    .size().rename(columns={'size': 'Nombre_Incendis', 'PeriodeQ3': 'Periode'})
)
totes_comarques = sorted(list(noms_geojson))
index_complet_q3 = pd.MultiIndex.from_product([labels_q3, totes_comarques], names=['Periode', 'Nom comarca'])
df_q3_comarca = (
    df_q3_comarca.set_index(['Periode', 'Nom comarca'])
    .reindex(index_complet_q3, fill_value=0).reset_index()
)

df_q3_comarca['Mitjana_focs_hist'] = df_q3_comarca.groupby('Nom comarca')['Nombre_Incendis'].transform("median")

df_q3_comarca['Dif_Mitjana_focs'] = df_q3_comarca['Nombre_Incendis'] - df_q3_comarca['Mitjana_focs_hist']

max_abs_p0 = float(df_q3_comarca[df_q3_comarca['Periode'] == labels_q3[0]]['Nombre_Incendis'].max())
lim_dif_ant = float(np.nanpercentile(df_q3_comarca['Dif_Mitjana_focs'].abs(), 96))

w3, h3 = mida_mapa(160)
mapes_q3 = []

for idx_m, p in enumerate(labels_q3):
    dades_p = df_q3_comarca[df_q3_comarca['Periode'] == p]
    p_ant = labels_q3[idx_m - 1]
    mostra_leg_dif = (idx_m == len(labels_q3) - 1)
    submap = alt.Chart(comarques_geo).mark_geoshape(stroke='white', strokeWidth=0.6).transform_lookup(
        lookup='properties.NOMCOMAR',
        from_=alt.LookupData(
            dades_p, 'Nom comarca',
            ['Nombre_Incendis', 'Mitjana_focs_hist', 'Dif_Mitjana_focs']
        )
    ).encode(
        color=alt.Color(
            'Dif_Mitjana_focs:Q',
            scale=alt.Scale(
                domain=[-lim_dif_ant, 0, lim_dif_ant],
                range=['#542788', '#f7f7f7', '#e66101'],
                clamp=True
            ),
            title="Variació vs. mitjana històrica",
            legend=alt.Legend(orient='bottom', gradientLength=200, titleFontSize=11, labelFontSize=10)
            if mostra_leg_dif else None
        ),
        tooltip=[
            alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
            alt.Tooltip('Nombre_Incendis:Q', title=f"Incendis ({p})"),
            alt.Tooltip("Mitjana_focs_hist:Q", title="Mitjana històrica", format='.2f')
            # alt.Tooltip('Incendis_Anterior:Q', title=f"Incendis trienni previ ({p_ant})", format=','),
            # alt.Tooltip('Dif_Mitjana_focs:Q', title=f"Diferència ({p} vs {p_ant})", format='+,d')
        ]
    ).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
        width=w3, height=h3,
        title=alt.TitleParams(f"{p}", fontSize=14, fontWeight='bold'))

    mapes_q3.append(submap)

q3_final = alt.concat(*mapes_q3, columns=9, spacing=12).resolve_scale(color='independent').properties(
    title=titol_rt(
        "Q3 · Evolució triennal per comarca: variació absoluta respecte la mediana històrica de focs (1998-2024)",
        ""
    ),
    ).configure_view(stroke=None)

q3_final
q3_final.save('outputs/Q3.json')
q3_final.save('outputs/Q3.html')


In [ ]:
w3, h3 = mida_mapa(275)

for m in mapes_q3:
    m.width = w3
    m.height = h3

q3_final_v2 = alt.concat(*mapes_q3, columns=3, spacing=10).resolve_scale(color='independent').properties(
    title=titol_rt(
        "Q3 · Evolució triennal per comarca: variació absoluta respecte la mediana històrica de focs (1998-2024)",
        ""
    ),
).configure_view(stroke=None)

q3_final_v2
q3_final.save('outputs/Q3_3x3.json')
q3_final_v2.save('outputs/Q3_3x3.html')

In [ ]:
# %% ===================================================================
# Q4 FINAL (trienni) · Hectàrees cremades (barra mirall) + Superfície Forestal (donut)
#                       + Nombre d'incendis per magnitud (boles per columnes)
# Versió 10: eliminat l'espai mort, anotació Agramunt entre 2004-2006 i
#            2007-2009 (sobre Trieni -> donut), total focs visible a 1998-2000
#            i llegenda Forestal/No Forestal restaurada al peu.
# =======================================================================
# SUPOSICIONS: aquestes variables ja existeixen de cel·les anteriors:
#   - df             -> 'Any', 'SuperficieTotal', 'ClasseMida',
#                        'SuperficieTotalForestal', 'SuperficieTotalNoForestal',
#                        'Nom municipi', 'Nom comarca'
#   - classes_mida   -> ['<1 ha', '1-10 ha', '10-100 ha', '100-500 ha', '>500 ha']
#   - mida_scale     -> alt.Scale(domain=classes_mida, scheme=..., reverse=True)
# -----------------------------------------------------------------------
import math
import pandas as pd
import altair as alt

# ---------------------------------------------------------------------
# 0) TRIENNIS
# ---------------------------------------------------------------------
ANY_MIN = int(df['Any'].min())
MIDA_PERIODE = 3


def assigna_trienni(any_valor, any_min=ANY_MIN, mida=MIDA_PERIODE):
    idx = (any_valor - any_min) // mida
    inici = any_min + idx * mida
    fi = inici + mida - 1
    return f"{inici}-{fi}"


df['PeriodeQ4'] = df['Any'].apply(assigna_trienni)
labels_q4 = sorted(df['PeriodeQ4'].unique(), key=lambda p: int(p.split('-')[0]))
periode_to_idx = {p: i for i, p in enumerate(labels_q4)}
N_PERIODES = len(labels_q4)

tot_focs_df = (
    df.groupby('PeriodeQ4', as_index=False)
    .size()
    .rename(columns={'PeriodeQ4': 'Periode', 'size': 'Total_Focs'})
)
tot_focs_map = dict(zip(tot_focs_df['Periode'], tot_focs_df['Total_Focs']))

# ---------------------------------------------------------------------
# 1) HECTÀREES CREMADES per classe i trienni — ORDRE DESCENDENT
# ---------------------------------------------------------------------
classes_mida_desc = list(reversed(classes_mida))

df_q4_mida_raw = (
    df.groupby(['PeriodeQ4', 'ClasseMida'], as_index=False, observed=True)
    .agg(Superficie=('SuperficieTotal', 'sum'), NumFocs=('SuperficieTotal', 'count'))
    .rename(columns={'PeriodeQ4': 'Periode'})
)

parts_abs = []
for idx_p, p in enumerate(labels_q4):
    sub = (
        df_q4_mida_raw[df_q4_mida_raw['Periode'] == p]
        .set_index('ClasseMida').reindex(classes_mida_desc).reset_index()
    )
    sub['Superficie'] = sub['Superficie'].fillna(0)
    sub['NumFocs'] = sub['NumFocs'].fillna(0)
    tot = sub['Superficie'].sum()
    tot_focs_p = float(tot_focs_map.get(p, 0) or 0)

    sub['pct'] = sub['Superficie'] / tot if tot else 0
    sub['pct_focs'] = sub['NumFocs'] / tot_focs_p if tot_focs_p else 0.0
    sub['x1_abs'] = sub['Superficie'].cumsum()
    sub['x0_abs'] = sub['x1_abs'] - sub['Superficie']
    sub['x_mig_abs'] = (sub['x0_abs'] + sub['x1_abs']) / 2
    sub['Periode'] = p
    sub['y_idx'] = idx_p
    sub['Total_Periode'] = tot
    sub['label_pct'] = sub['pct'].apply(
        lambda v: (f"{round(v * 100)}%" if idx_p == 0 else f"{round(v * 100)}") if v >= 0.05 else ""
    )
    parts_abs.append(sub)
df_q4_mida_abs = pd.concat(parts_abs, ignore_index=True)

max_ha_periode = df_q4_mida_abs.groupby('Periode')['x1_abs'].max().max()
totals_periode = df_q4_mida_abs.groupby('Periode', as_index=False).agg(
    Total_Periode=('Total_Periode', 'first'),
    y_idx=('y_idx', 'first')
)
totals_periode['label_total'] = totals_periode['Total_Periode'].apply(lambda v: f"{v:,.0f} ha")
totals_periode_map = dict(zip(totals_periode['Periode'], totals_periode['Total_Periode']))

STEP_X_ABS = 5000
max_x_domain = max_ha_periode * 1.1
x_ticks_abs = list(range(0, int(max_x_domain) + 1, STEP_X_ABS))
X_SCALE_ABS = alt.Scale(domain=[max_x_domain, 0], nice=False)

# ---------------------------------------------------------------------
# 2) SUPERFÍCIE FORESTAL vs NO FORESTAL — colors directes
# ---------------------------------------------------------------------
df_q4_tipus_raw = (
    df.groupby('PeriodeQ4', as_index=False, observed=True)
    .agg(Forestal=('SuperficieTotalForestal', 'sum'), NoForestal=('SuperficieTotalNoForestal', 'sum'))
    .rename(columns={'PeriodeQ4': 'Periode'})
    .melt(id_vars='Periode', value_vars=['Forestal', 'NoForestal'], var_name='Tipus', value_name='Superficie')
)
tipus_domain = ['Forestal', 'NoForestal']
tipus_colors = {'Forestal': '#1b7837', 'NoForestal': '#d73027'}

OFFSET_RAD = 0

parts_tipus = []
for idx_p, p in enumerate(labels_q4):
    sub = df_q4_tipus_raw[df_q4_tipus_raw['Periode'] == p].set_index('Tipus').reindex(tipus_domain).reset_index()
    sub['Superficie'] = sub['Superficie'].fillna(0)
    tot = sub['Superficie'].sum()
    sub['frac'] = sub['Superficie'] / tot if tot else 0
    sub['cum_end'] = sub['frac'].cumsum()
    sub['cum_start'] = sub['cum_end'] - sub['frac']
    sub['theta0'] = OFFSET_RAD + sub['cum_start'] * 2 * math.pi
    sub['theta1'] = OFFSET_RAD + sub['cum_end'] * 2 * math.pi
    sub['theta_mid'] = (sub['theta0'] + sub['theta1']) / 2
    sub['Periode'] = p
    sub['y_idx'] = idx_p
    sub['color_hex'] = sub['Tipus'].map(tipus_colors)
    sub['label_abs'] = sub.apply(
        lambda r: f"{round(r['frac'] * 100)}%" if r['Tipus'] == 'Forestal' else "", axis=1
    )
    parts_tipus.append(sub)
df_q4_tipus = pd.concat(parts_tipus, ignore_index=True)

ann_0709_text = ""
sub_0709 = df[df['PeriodeQ4'] == '2007-2009']
if len(sub_0709) > 0:
    top_nf = sub_0709.loc[sub_0709['SuperficieTotalNoForestal'].idxmax()]
    mun_nf = top_nf.get('Nom municipi', top_nf.get('Nom comarca', ''))
    if pd.isna(mun_nf) or str(mun_nf).strip() == '':
        mun_nf = top_nf.get('Nom comarca', '')
    any_nf = int(top_nf['Any'])
    ha_nf = float(top_nf['SuperficieTotalNoForestal'])
    ann_0709_text = f"GIF {mun_nf} ({any_nf}):\n{ha_nf:,.0f} ha agrícoles"

# ---------------------------------------------------------------------
# 3) AMPLADES I ALÇADES (SENSE ESPAI MORT)
# ---------------------------------------------------------------------
ALCADA_BARRA = 52
ALTURA_FILA = ALCADA_BARRA + 8
TOTAL_HEIGHT = N_PERIODES * ALTURA_FILA

W_BAR_ABS = 870
W_ANY_Q4 = 95      
W_DONUT_PIE = 64
W_BOLES_COMBO = 480 # una mica més d'aire a la dreta perquè hi càpiga "2,xxx focs" de 1998-2000
ESPAI = 8
TOTAL_WIDTH = W_BAR_ABS + ESPAI + W_ANY_Q4 + ESPAI + W_DONUT_PIE + ESPAI + W_BOLES_COMBO

Y_SCALE_GLOBAL = alt.Scale(domain=[-0.5, N_PERIODES - 0.5], reverse=True, nice=False)

MIDA_ANOTACIO = 13


# ---------------------------------------------------------------------
# 4) BOLES PER COLUMNES (5 files/col) + TOOLTIP ENRIQUIT
# ---------------------------------------------------------------------
GROUP_SIZE_Q4 = 10
N_FILES_BOLES = 5
classes_resta = [c for c in classes_mida if c != '>500 ha']

max_gifs = max(int((df[df['PeriodeQ4'] == p]['SuperficieTotal'] >= 500).sum()) for p in labels_q4)
COLS_GIF_RESERVA = max(2, math.ceil(max_gifs / N_FILES_BOLES))
GAP_ENTRE_BLOCS = 0.8
START_COL_RESTA = COLS_GIF_RESERVA + GAP_ENTRE_BLOCS
X_DIVISOR_BOLES = ((COLS_GIF_RESERVA - 1) + START_COL_RESTA) / 2


def boles_columnes_periode(subdf, periode):
    files = []
    idx_p = periode_to_idx[periode]
    tot_p = float(totals_periode_map.get(periode, 0) or 0)
    tot_focs_p = float(tot_focs_map.get(periode, 0) or 0)

    grans = subdf[subdf['SuperficieTotal'] >= 500].sort_values('SuperficieTotal', ascending=False)
    for idx_gif, (_, r) in enumerate(grans.iterrows()):
        mun = r.get('Nom municipi', r.get('Nom comarca', ''))
        if pd.isna(mun) or str(mun).strip() == '':
            mun = r.get('Nom comarca', '')
        ha_tot = float(r['SuperficieTotal'])
        ha_for = float(r.get('SuperficieTotalForestal', 0) or 0)
        ha_nofor = float(r.get('SuperficieTotalNoForestal', 0) or 0)
        fila_sub = idx_gif % N_FILES_BOLES

        files.append({
            'Periode': periode,
            'TipusBola': 'GIF',
            'ClasseMida': '>500 ha',
            'col': idx_gif // N_FILES_BOLES,
            'y_ball': idx_p + (fila_sub - (N_FILES_BOLES - 1) / 2) * 0.185,
            'Municipi': str(mun),
            'Nom_Comarca': r['Nom comarca'],
            'Any': int(r['Any']),
            'ha_real': ha_tot,
            'ha_forestal': ha_for,
            'ha_no_forestal': ha_nofor,
            'pct_forestal': (ha_for / ha_tot) if ha_tot > 0 else 0.0,
            'pct_no_forestal': (ha_nofor / ha_tot) if ha_tot > 0 else 0.0,
            'pct_trienni': (ha_tot / tot_p) if tot_p > 0 else 0.0,
            'Incendis_Reals': 1,
            'pct_focs_classe': (1.0 / tot_focs_p) if tot_focs_p > 0 else 0.0,
            'Equivalencia': '1 bola = 1 Gran Incendi Forestal (>500 ha)'
        })

    idx_resta = 0
    for classe in reversed(classes_resta):
        n = int((subdf['ClasseMida'] == classe).sum())
        if n > 0 and classe == '100-500 ha':
            n_boles = max(1, int(round(n / GROUP_SIZE_Q4)))
        else:
            n_boles = int(n // GROUP_SIZE_Q4)
        pct_cls = (n / tot_focs_p) if tot_focs_p > 0 else 0.0
        for _ in range(n_boles):
            fila_sub = idx_resta % N_FILES_BOLES
            files.append({
                'Periode': periode,
                'TipusBola': 'Resta',
                'ClasseMida': classe,
                'col': START_COL_RESTA + (idx_resta // N_FILES_BOLES),
                'y_ball': idx_p + (fila_sub - (N_FILES_BOLES - 1) / 2) * 0.185,
                'Municipi': '',
                'Nom_Comarca': '',
                'Any': 0,
                'ha_real': 0.0,
                'ha_forestal': 0.0,
                'ha_no_forestal': 0.0,
                'pct_forestal': 0.0,
                'pct_no_forestal': 0.0,
                'pct_trienni': 0.0,
                'Incendis_Reals': n,
                'pct_focs_classe': pct_cls,
                'Equivalencia': f"1 bola = {GROUP_SIZE_Q4} incendis ({classe})"
            })
            idx_resta += 1

    return pd.DataFrame(files)


df_boles = pd.concat(
    [boles_columnes_periode(df[df['PeriodeQ4'] == p], p) for p in labels_q4],
    ignore_index=True
)
max_col_boles = max(float(df_boles['col'].max()) if len(df_boles) else 20.0, START_COL_RESTA + 1)

boles_max_col = df_boles.groupby('Periode', as_index=False).agg(max_col=('col', 'max'))
boles_max_col['y_idx'] = boles_max_col['Periode'].map(periode_to_idx)
boles_max_col['Total_Focs'] = boles_max_col['Periode'].map(tot_focs_map)
boles_max_col['label_focs'] = boles_max_col['Total_Focs'].apply(lambda v: f"{int(v):,} focs")

# ---------------------------------------------------------------------
# 5) HELPER PER A FLETXES CORBADES AMB PUNTA EN `mark_rule`
# ---------------------------------------------------------------------
def bezier_fletxa_rules(x0, y0, cx, cy, x1, y1, x_domain, y_domain, width_px, height_px, reverse_x=False, reverse_y=True, n=24):
    pts = []
    for i in range(n + 1):
        t = i / n
        x = (1 - t) ** 2 * x0 + 2 * (1 - t) * t * cx + t ** 2 * x1
        y = (1 - t) ** 2 * y0 + 2 * (1 - t) * t * cy + t ** 2 * y1
        pts.append({'x': x, 'y': y, 't': t})
    df_line = pd.DataFrame(pts)

    x_span = abs(x_domain[1] - x_domain[0])
    y_span = abs(y_domain[1] - y_domain[0])
    dx_data = pts[-1]['x'] - pts[-2]['x']
    dy_data = pts[-1]['y'] - pts[-2]['y']
    dx_px = (-dx_data if reverse_x else dx_data) / x_span * width_px
    dy_px = (dy_data if reverse_y else -dy_data) / y_span * height_px
    norm = math.hypot(dx_px, dy_px) or 1.0
    ux, uy = dx_px / norm, dy_px / norm

    L = 7.0
    c, s = math.cos(math.radians(28)), math.sin(math.radians(28))
    wings_px = [
        (L * (-ux * c + uy * s), L * (-ux * s - uy * c)),
        (L * (-ux * c - uy * s), L * (ux * s - uy * c)),
    ]
    wings_rows = []
    for wx, wy in wings_px:
        dx_w = (-wx if reverse_x else wx) * x_span / width_px
        dy_w = (wy if reverse_y else -wy) * y_span / height_px
        wings_rows.append({'x': x1 + dx_w, 'y': y1 + dy_w, 'x2': x1, 'y2': y1})

    return df_line, pd.DataFrame(wings_rows)


ESTIL_TITOL = dict(fontSize=13, fontWeight='bold', align='center', color='#111111')


def _spacer(width, height=1):
    return alt.Chart(pd.DataFrame({'t': ['']})).mark_text().encode(text='t:N').properties(width=width, height=height)


# ---------------------------------------------------------------------
# 6) LLEGENDA ÚNICA (5 classes) + CAPÇALERA DE TÍTOLS
# ---------------------------------------------------------------------
leg_df = pd.DataFrame({'ClasseMida': classes_mida, 'x': range(len(classes_mida))})
base_leg = alt.Chart(leg_df).encode(
    x=alt.X('x:Q', axis=None, scale=alt.Scale(domain=[-1.8, len(classes_mida) + 0.8], nice=False)),
    y=alt.value(14)
)
leg_dots = base_leg.mark_circle(size=200, stroke='white', strokeWidth=0.6).encode(
    color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None)
)
leg_text = base_leg.mark_text(
    align='left', dx=12, fontSize=12, fontWeight='bold', color='#111111'
).encode(text='ClasseMida:N')
llegenda_unica = (leg_dots + leg_text).properties(width=TOTAL_WIDTH, height=28)

tit_bar = alt.Chart(pd.DataFrame({'t': ["Hectàrees cremades segons magnitud de l'incendi"]})).mark_text(
    **ESTIL_TITOL
).encode(text='t:N').properties(width=W_BAR_ABS, height=1)

tit_any = alt.Chart(pd.DataFrame({'t': ['Trieni']})).mark_text(**ESTIL_TITOL).encode(text='t:N').properties(
    width=W_ANY_Q4, height=1
)

tit_donut = alt.Chart(pd.DataFrame({'t': ['Superfície Forestal cremada']})).mark_text(**ESTIL_TITOL).encode(
    text='t:N'
).properties(width=W_DONUT_PIE, height=1)

tit_boles = alt.Chart(pd.DataFrame({'t': ["Nombre d'incendis per magnitud"]})).mark_text(**ESTIL_TITOL).encode(
    text='t:N'
).properties(width=W_BOLES_COMBO-50, height=1)

fila_titols = alt.hconcat(tit_bar, tit_any, tit_donut, tit_boles, spacing=ESPAI)

# ---------------------------------------------------------------------
# 7) LES 4 COLUMNES
# ---------------------------------------------------------------------
# --- Columna 1: BARRES ---
bar_layers = [
    alt.Chart(df_q4_mida_abs).mark_bar(height=ALCADA_BARRA).encode(
        x=alt.X(
            'x0_abs:Q', scale=X_SCALE_ABS,
            axis=alt.Axis(title='Superfície total cremada (ha)', format=',.0f', values=x_ticks_abs)
        ),
        x2='x1_abs:Q',
        y=alt.Y('y_idx:Q', scale=Y_SCALE_GLOBAL, axis=None),
        color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
        tooltip=[
            alt.Tooltip('Periode:N', title='Trienni'),
            alt.Tooltip('ClasseMida:N', title='Magnitud'),
            alt.Tooltip('Superficie:Q', title='Hectàrees cremades (ha)', format=',.1f'),
            alt.Tooltip('pct:Q', title="% de superfície del trienni", format='.1%'),
            alt.Tooltip('NumFocs:Q', title="Nombre d'incendis", format=','),
            alt.Tooltip('pct_focs:Q', title="% d'incendis del trienni", format='.1%')
        ]
    ),
    alt.Chart(df_q4_mida_abs[df_q4_mida_abs['pct'] >= 0.05]).mark_text(
        color='white', fontSize=10.5, fontWeight='bold', baseline='middle'
    ).encode(
        x=alt.X('x_mig_abs:Q', scale=X_SCALE_ABS),
        y=alt.Y('y_idx:Q', scale=Y_SCALE_GLOBAL, axis=None),
        text='label_pct:N'
    ),
    alt.Chart(totals_periode).mark_text(
        align='right', dx=-6, fontSize=12, fontWeight='bold', color='#000000', baseline='middle'
    ).encode(
        x=alt.X('Total_Periode:Q', scale=X_SCALE_ABS),
        y=alt.Y('y_idx:Q', scale=Y_SCALE_GLOBAL, axis=None),
        text='label_total:N'
    )
]

# Anotació 1: Catalunya Central 1998
row_98 = df_q4_mida_abs[(df_q4_mida_abs['Periode'] == '1998-2000') & (df_q4_mida_abs['ClasseMida'] == '>500 ha')]
if len(row_98) > 0:
    x_target_98 = float(row_98['x_mig_abs'].iloc[0]) * 1.45
    df_txt_98 = pd.DataFrame([{
        'x': max_ha_periode * 0.80,
        'y': 2.05,
        't': "Grans incendis forestals campanya 1998\na la Catalunya Central"
    }])
    df_l98, df_w98 = bezier_fletxa_rules(
        x0=max_ha_periode * 0.65, y0=1.70,
        cx=max_ha_periode * 0.58, cy=1.15,
        x1=x_target_98, y1=0.46,
        x_domain=[max_x_domain, 0], y_domain=[-0.5, N_PERIODES - 0.5],
        width_px=W_BAR_ABS, height_px=TOTAL_HEIGHT, reverse_x=True, reverse_y=True
    )
    bar_layers.extend([
        alt.Chart(df_txt_98).mark_text(
            align='center', fontSize=MIDA_ANOTACIO, fontStyle='italic', color='#333333',
            lineBreak='\n', baseline='middle'
        ).encode(x=alt.X('x:Q', scale=X_SCALE_ABS), y=alt.Y('y:Q', scale=Y_SCALE_GLOBAL, axis=None), text='t:N'),
        alt.Chart(df_l98).mark_line(color='#444444', strokeWidth=1.2).encode(
            x=alt.X('x:Q', scale=X_SCALE_ABS), y=alt.Y('y:Q', scale=Y_SCALE_GLOBAL, axis=None), order='t:Q'
        ),
        alt.Chart(df_w98).mark_rule(color='#444444', strokeWidth=1.3).encode(
            x=alt.X('x:Q', scale=X_SCALE_ABS), x2='x2:Q',
            y=alt.Y('y:Q', scale=Y_SCALE_GLOBAL, axis=None), y2='y2:Q'
        )
    ])

# Anotació 2: Alt Empordà 2012
row_10 = df_q4_mida_abs[(df_q4_mida_abs['Periode'] == '2010-2012') & (df_q4_mida_abs['ClasseMida'] == '>500 ha')]
if len(row_10) > 0:
    x_target_10 = float(row_10['x_mig_abs'].iloc[0]) * 1.65
    df_txt_10 = pd.DataFrame([{
        'x': max_ha_periode * 0.72,
        'y': 5.05,
        't': "Principalment pel Gran Incendi Forestal\nde l'Alt Empordà (2012)"
    }])
    df_l10, df_w10 = bezier_fletxa_rules(
        x0=max_ha_periode * 0.54, y0=5.10,
        cx=max_ha_periode * 0.38, cy=5.35,
        x1=x_target_10, y1=4.46,
        x_domain=[max_x_domain, 0], y_domain=[-0.5, N_PERIODES - 0.5],
        width_px=W_BAR_ABS, height_px=TOTAL_HEIGHT, reverse_x=True, reverse_y=True
    )
    bar_layers.extend([
        alt.Chart(df_txt_10).mark_text(
            align='center', fontSize=MIDA_ANOTACIO, fontStyle='italic', color='#333333',
            lineBreak='\n', baseline='middle'
        ).encode(x=alt.X('x:Q', scale=X_SCALE_ABS), y=alt.Y('y:Q', scale=Y_SCALE_GLOBAL, axis=None), text='t:N'),
        alt.Chart(df_l10).mark_line(color='#444444', strokeWidth=1.2).encode(
            x=alt.X('x:Q', scale=X_SCALE_ABS), y=alt.Y('y:Q', scale=Y_SCALE_GLOBAL, axis=None), order='t:Q'
        ),
        alt.Chart(df_w10).mark_rule(color='#444444', strokeWidth=1.3).encode(
            x=alt.X('x:Q', scale=X_SCALE_ABS), x2='x2:Q',
            y=alt.Y('y:Q', scale=Y_SCALE_GLOBAL, axis=None), y2='y2:Q'
        )
    ])

c_bar = alt.layer(*bar_layers).properties(width=W_BAR_ABS, height=TOTAL_HEIGHT)

# --- Columna 2: TRIENI + ANOTACIÓ AGRAMUNT JUST ENTRE 2004-2006 I 2007-2009 ---
X_SCALE_ANY = alt.Scale(domain=[0, W_ANY_Q4], nice=False)
df_any_all = pd.DataFrame({'Periode': labels_q4, 'y_idx': range(N_PERIODES), 'x': W_ANY_Q4 / 2})

any_layers = [
    alt.Chart(df_any_all).mark_text(
        fontSize=16, fontWeight='bold', align='center', color='#111111'
    ).encode(
        x=alt.X('x:Q', scale=X_SCALE_ANY, axis=None),
        y=alt.Y('y_idx:Q', scale=Y_SCALE_GLOBAL, axis=None),
        text='Periode:N'
    )
]

if ann_0709_text:
    df_box_any = pd.DataFrame([{
        'xc': W_ANY_Q4 / 2, 'yc': 2.38,
        't': ann_0709_text
    }])
    df_ld, df_wd = bezier_fletxa_rules(
        x0=W_ANY_Q4 - 8, y0=2.72,
        cx=W_ANY_Q4 + 4, cy=2.84,
        x1=W_ANY_Q4 + 12, y1=2.92,
        x_domain=[0, W_ANY_Q4], y_domain=[-0.5, N_PERIODES - 0.5],
        width_px=W_ANY_Q4, height_px=TOTAL_HEIGHT, reverse_x=False, reverse_y=True
    )
    any_layers.extend([
        alt.Chart(df_box_any).mark_text(
            align='center', fontSize=MIDA_ANOTACIO, fontStyle='italic', color='#333333',
            lineBreak='\n', baseline='middle'
        ).encode(
            x=alt.X('xc:Q', scale=X_SCALE_ANY, axis=None),
            y=alt.Y('yc:Q', scale=Y_SCALE_GLOBAL, axis=None),
            text='t:N'
        ),
        alt.Chart(df_ld).mark_line(color='#444444', strokeWidth=1.1, clip=False).encode(
            x=alt.X('x:Q', scale=X_SCALE_ANY, axis=None),
            y=alt.Y('y:Q', scale=Y_SCALE_GLOBAL, axis=None),
            order='t:Q'
        ),
        alt.Chart(df_wd).mark_rule(color='#444444', strokeWidth=1.2, clip=False).encode(
            x=alt.X('x:Q', scale=X_SCALE_ANY, axis=None), x2='x2:Q',
            y=alt.Y('y:Q', scale=Y_SCALE_GLOBAL, axis=None), y2='y2:Q'
        )
    ])

c_any = alt.layer(*any_layers).properties(width=W_ANY_Q4, height=TOTAL_HEIGHT)

# --- Columna 3: DONUTS NATIUS 
donuts_pies = []
for p in labels_q4:
    sub_t = df_q4_tipus[df_q4_tipus['Periode'] == p]
    arc = alt.Chart(sub_t).mark_arc(innerRadius=5, outerRadius=26, stroke='white', strokeWidth=1).encode(
        theta=alt.Theta('theta0:Q', scale=None),
        theta2=alt.Theta2('theta1:Q'),
        color=alt.Color('color_hex:N', scale=None, legend=None),
        tooltip=[
            alt.Tooltip('Periode:N', title='Trienni'),
            alt.Tooltip('Tipus:N', title='Cobertura'),
            alt.Tooltip('Superficie:Q', title='Hectàrees (ha)', format=',.1f'),
            alt.Tooltip('frac:Q', title='Percentatge', format='.1%')
        ]
    )
    text_arc = alt.Chart(sub_t[sub_t['label_abs'] != ""]).mark_text(
        radius=18, fontSize=12, fontWeight='bold', color='white'
    ).encode(theta=alt.Theta('theta_mid:Q', scale=None), text='label_abs:N')
    donuts_pies.append((arc + text_arc).properties(width=W_DONUT_PIE, height=ALTURA_FILA))

col_donut_pies = alt.vconcat(*donuts_pies, spacing=0)

# --- Columna 4: BOLES + SEPARADOR SUBTIL + TOTAL DE FOCS A LA DRETA ---
X_SCALE_BOLES = alt.Scale(domain=[-0.6, max_col_boles + 11.0], nice=False)
x_boles_enc = alt.X('col:Q', scale=X_SCALE_BOLES, axis=None)
y_boles_enc = alt.Y('y_ball:Q', scale=Y_SCALE_GLOBAL, axis=None)

sub_bg_all = df_boles[df_boles['TipusBola'] == 'GIF']
sub_br_all = df_boles[df_boles['TipusBola'] == 'Resta']

layers_boles = [
    alt.Chart(pd.DataFrame([{'x': X_DIVISOR_BOLES}])).mark_rule(
        color='#cccccc', strokeDash=[3, 3], strokeWidth=1
    ).encode(x=alt.X('x:Q', scale=X_SCALE_BOLES, axis=None))
]

if len(sub_bg_all) > 0:
    layers_boles.append(
        alt.Chart(sub_bg_all).mark_circle(
            size=64, opacity=0.95, stroke='white', strokeWidth=0.5
        ).encode(
            x=x_boles_enc,
            y=y_boles_enc,
            color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
            tooltip=[
                alt.Tooltip('Municipi:N', title='Municipi'),
                alt.Tooltip('Nom_Comarca:N', title='Comarca'),
                alt.Tooltip('Any:Q', title='Any'),
                alt.Tooltip('ha_real:Q', title='Superfície total (ha)', format=',.1f'),
                alt.Tooltip('pct_trienni:Q', title='Pes sobre el total del trienni', format='.1%'),
                alt.Tooltip('pct_focs_classe:Q', title="% d'incendis del trienni", format='.1%'),
                alt.Tooltip('ha_forestal:Q', title='Superfície forestal (ha)', format=',.1f'),
                alt.Tooltip('pct_forestal:Q', title='% Forestal', format='.1%'),
                alt.Tooltip('ha_no_forestal:Q', title='Superfície no forestal (ha)', format=',.1f'),
                alt.Tooltip('pct_no_forestal:Q', title='% No Forestal', format='.1%')
            ]
        )
    )
if len(sub_br_all) > 0:
    layers_boles.append(
        alt.Chart(sub_br_all).mark_circle(
            size=64, opacity=0.95, stroke='white', strokeWidth=0.4
        ).encode(
            x=x_boles_enc,
            y=y_boles_enc,
            color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
            tooltip=[
                alt.Tooltip('Periode:N', title='Trienni'),
                alt.Tooltip('ClasseMida:N', title='Categoria'),
                alt.Tooltip('Incendis_Reals:Q', title="Total incendis d'aquesta classe", format=','),
                alt.Tooltip('pct_focs_classe:Q', title="% d'incendis del trienni", format='.1%'),
                alt.Tooltip('Equivalencia:N', title='Escala')
            ]
        )
    )

layers_boles.append(
    alt.Chart(boles_max_col).mark_text(
        align='left', dx=10, fontSize=11.5, fontWeight='bold', color='#111111', baseline='middle', clip=False
    ).encode(
        x=alt.X('max_col:Q', scale=X_SCALE_BOLES, axis=None),
        y=alt.Y('y_idx:Q', scale=Y_SCALE_GLOBAL, axis=None),
        text='label_focs:N'
    )
)

c_boles = alt.layer(*layers_boles).properties(width=W_BOLES_COMBO, height=TOTAL_HEIGHT)

cos_grafic = alt.hconcat(c_bar, c_any, col_donut_pies, c_boles, spacing=ESPAI)

# ---------------------------------------------------------------------
# 8) LLEGENDA INFERIOR: Forestal / No Forestal (sota donut) + Escala de boles
# ---------------------------------------------------------------------
lleg_tipus_df = pd.DataFrame({
    'color_hex': [tipus_colors['Forestal'], tipus_colors['NoForestal']],
    'label': ['Forestal', 'No Forestal'],
    'y': [0, 1]
})
base_t = alt.Chart(lleg_tipus_df).encode(
    y=alt.Y('y:Q', axis=None, scale=alt.Scale(domain=[-0.7, 1.7], reverse=True))
)
dot_t = base_t.mark_circle(size=130, stroke='white', strokeWidth=0.6).encode(
    x=alt.value(4),
    color=alt.Color('color_hex:N', scale=None, legend=None)
)
text_t = base_t.mark_text(align='left', fontSize=11.5, fontWeight='bold', color='#111111').encode(
    x=alt.value(16), text='label:N'
)
llegenda_tipus_sota = (dot_t + text_t).properties(width=W_DONUT_PIE, height=38)

leg_boles_gif = alt.Chart(pd.DataFrame({
    'ClasseMida': ['>500 ha'],
    'label': ['Bloc esquerre (>500 ha) = 1 incendi (GIF)']
})).encode(y=alt.value(10))
dot_bg = leg_boles_gif.mark_circle(size=64, stroke='white', strokeWidth=0.5).encode(
    x=alt.value(16),
    color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None)
)
txt_bg = leg_boles_gif.mark_text(align='left', fontSize=11, color='#222222').encode(
    x=alt.value(28), text='label:N'
)

leg_boles_petita = alt.Chart(pd.DataFrame({
    'label': [f'Bloc dret (<500 ha) = {GROUP_SIZE_Q4} incendis']
})).encode(y=alt.value(27))
dot_bp = leg_boles_petita.mark_circle(size=64, color='#666666', stroke='white', strokeWidth=0.4).encode(
    x=alt.value(16)
)
txt_bp = leg_boles_petita.mark_text(align='left', fontSize=11, color='#222222').encode(
    x=alt.value(28), text='label:N'
)
llegenda_boles_sota = (dot_bg + txt_bg + dot_bp + txt_bp).properties(width=W_BOLES_COMBO, height=38)

fila_llegenda_peu = alt.hconcat(
    _spacer(W_BAR_ABS, 38),
    _spacer(W_ANY_Q4, 38),
    llegenda_tipus_sota,
    llegenda_boles_sota,
    spacing=ESPAI+2
)

# ---------------------------------------------------------------------
# 9) COMPOSICIÓ FINAL
# ---------------------------------------------------------------------
q4_final_trienni = (
    alt.vconcat(llegenda_unica, fila_titols, cos_grafic, fila_llegenda_peu, spacing=3)
    .properties(
        title=alt.TitleParams(
            "Q4 Evolució de la distribució dels incendis forestals a Catalunya per trieni (1998-2024)",
            fontSize=16, anchor='middle'
        )
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=False)
)

q4_final_trienni.save('outputs/Q4.json')
q4_final_trienni.save('outputs/Q4.html')